# Corp Income — OCR & kiểm tra pháp lý doanh nghiệp (BRD mục 1–4)

**Cách dùng:** chạy lần lượt từ trên xuống (hoặc *Run all*). Chỉ cần sửa **2 ô**:

1. **Ô ② Cài đặt LLM** — chọn nhà cung cấp, model và khoá (dùng `dbutils.secrets`).
2. **Ô ④ Hồ sơ** — đường dẫn thư mục hồ sơ KHDN, ngày vay, tên KH cá nhân vay vốn.

Không cần prompt. Kết quả: `report.md`, `corp_income.xlsx`, `result.json`, và thư mục `ocr/` (văn bản đọc được của từng file).

Cách xếp hồ sơ vào thư mục: xem `docs/DATABRICKS.md` mục *Hồ sơ đầu vào*.

In [ ]:
# ① Cài thư viện — chạy 1 lần mỗi khi cluster khởi động (Databricks).
#    Chạy local thì bỏ ô này: pip install -r requirements.txt
%pip install -q -r requirements.txt
dbutils.library.restartPython()

In [ ]:
# ② Cài đặt LLM — CHỈ SỬA Ô NÀY (phần LLM)
import os, sys
from pathlib import Path

PROJECT_ROOT = Path().resolve()          # thư mục repo (nơi đặt notebook này)
sys.path.insert(0, str(PROJECT_ROOT))

from corp_income import Settings

# Chọn MỘT trong ba cách, xoá/ghi chú hai cách còn lại.

# --- Cách A: Anthropic API key ------------------------------------------------
settings = Settings.load(
    llm_provider="anthropic",
    llm_model="<tên model Claude, ví dụ model ID bạn được cấp>",
    anthropic_api_key=dbutils.secrets.get("<scope>", "<anthropic-key-name>"),
    # anthropic_base_url="https://<gateway nội bộ nếu có>",
)

# --- Cách B: Model serving endpoint trong Databricks ---------------------------
# _ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
# settings = Settings.load(
#     llm_provider="databricks",
#     llm_model="<tên serving endpoint>",
#     databricks_host=_ctx.apiUrl().get(),
#     databricks_token=_ctx.apiToken().get(),
# )

# --- Cách C: Gateway nội bộ chuẩn OpenAI ---------------------------------------
# settings = Settings.load(
#     llm_provider="openai",
#     llm_model="<tên model>",
#     openai_api_key=dbutils.secrets.get("<scope>", "<key-name>"),
#     openai_base_url="https://<gateway>/v1",
# )

# OCR (Tesseract đã cài bằng scripts/install_tesseract.sh)
os.environ.setdefault("OCR_LANG", "vie+eng")
# Giữ kết quả OCR giữa các lần chạy (đỡ OCR lại): đặt vào một Volume
# os.environ["OCR_CACHE_DIR"] = "/Volumes/<catalog>/<schema>/<volume>/ocr_cache"

for key, value in settings.describe().items():
    print(f"{key:20} {value}")

In [ ]:
# ③ Kiểm tra kết nối — 1 lệnh gọi nhỏ để chắc chắn key, URL, model đều đúng
from corp_income import build_llm, check_connection, LLMConfigError

try:
    llm = build_llm(settings)
    print("Kết nối OK:", check_connection(llm))
except LLMConfigError as exc:
    print("❌ Thiếu cài đặt — sửa ô ②:\n", exc)
    raise
except Exception as exc:
    print("❌ Gọi LLM thất bại:", type(exc).__name__, str(exc)[:500])
    print("   401/403 → key sai | 404 → model hoặc URL sai | Connection error → mạng/URL")
    raise

import shutil
print("Tesseract:", shutil.which("tesseract") or "❌ CHƯA CÀI — xem scripts/install_tesseract.sh")

In [ ]:
# ④ Hồ sơ — SỬA Ô NÀY cho mỗi bộ hồ sơ
DOSSIER = "/Volumes/<catalog>/<schema>/<volume>/khdn_abc"   # thư mục hồ sơ KHDN
OUTPUT_DIR = "/Volumes/<catalog>/<schema>/<volume>/logs"     # nơi lưu kết quả

LOAN_DATE = "13/10/2026"            # mốc tính 12 tháng (dd/mm/yyyy); None = hôm nay
BORROWER_NAMES = ["Nguyễn Văn A"]   # KH cá nhân vay vốn (BRD mục 2); [] nếu không có

# Tuỳ chọn — để trống nếu chưa có
TYPE_OVERRIDES = {}   # {"scan01.pdf": "gcn_dkdn"} khi một file không tự nhận ra được loại
WATCHLIST = []        # [{"name": "...", "tax_code": "...", "list": "AML"}]
INTERNAL_DATA = {}    # dữ liệu BCDE/BEP/DBX theo schema mục: {"legal_basic": {...}}
EXTERNAL_DATA = {}    # kết quả API Cổng ĐKDN/thuế/BHXH: {"legal_basic": {...}}

In [ ]:
# ⑤ Xem trước hồ sơ (chưa gọi LLM) — kiểm tra đủ file và đúng thư mục
import pandas as pd
from corp_income.documents import box_of, classify, discover

files, skipped = discover([DOSSIER])
preview = [{"file": f.name, "thư mục": box_of(f) or "(gốc)",
            "loại (đoán theo tên/thư mục)": classify(f.name, "", box_of(f), TYPE_OVERRIDES.get(f.name))["type"]}
           for f in files]
display(pd.DataFrame(preview))
for message in skipped:
    print("Bỏ qua:", message)
print("'khac' ở đây chưa chắc là lỗi: lúc chạy thật, nội dung file cũng được dùng để phân loại.")

In [ ]:
# ⑥ Chạy — đọc/OCR → phân loại → trích xuất (LLM) → kiểm tra BRD mục 1–4
from corp_income import run_corp_income

result = run_corp_income(
    llm, [DOSSIER],
    loan_date=LOAN_DATE,
    borrower_names=BORROWER_NAMES,
    type_overrides=TYPE_OVERRIDES,
    watchlist=WATCHLIST,
    internal_data=INTERNAL_DATA,
    external_data=EXTERNAL_DATA,
    settings=settings,
)

print(f"Xong trong {result.meta['seconds']}s — {result.meta['files']} file\n")
for number, item in result.items.items():
    print(f"{number}. {item['title']}\n   → {item['conclusion']}\n")
print("--- Các bước ---")
for step in result.steps:
    print("-", step)

In [ ]:
# ⑦ Lưu kết quả
from datetime import datetime

run_dir = Path(OUTPUT_DIR) / f"{Path(DOSSIER).name}_{datetime.now():%Y%m%d_%H%M%S}"
paths = result.save(run_dir)
for name, path in paths.items():
    print(f"{name:7} {path}")

In [ ]:
# ⑧ Xem báo cáo
from IPython.display import Markdown
display(Markdown(result.markdown))

In [ ]:
# ⑨ Soát lỗi: OCR đọc được gì, LLM trích xuất được gì
docs = pd.DataFrame([{k: d.get(k) for k in ("filename", "box", "type", "method", "read_method", "chars",
                                             "passes_done", "errors")} for d in result.documents])
display(docs)

rows = [{"file": r["document"], "mục": r["pass"], "nguồn": r["source"],
         **{k: v for k, v in r["data"].items() if not isinstance(v, (list, dict))}}
        for r in result.records]
display(pd.DataFrame(rows))

# Văn bản OCR của một file (đổi tên file để xem file khác)
FILE = result.documents[0]["filename"] if result.documents else ""
print(next((d["content"][:3000] for d in result.documents if d["filename"] == FILE), ""))